<a href="https://colab.research.google.com/github/Adnan0425ninja/Student-Mark-Analyzer/blob/main/Student_Performance_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Performance Analytics Engine
**Author: Md Adnan Ashraf**

This notebook analyses a CSV dataset with Pandas, NumPy and Matplotlib. The included dataset is synthetic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SUBJECTS = ["Python", "Java", "DBMS", "Mathematics"]
PASS_MARK = 40
ATTENDANCE_THRESHOLD = 75


## 1. Load the dataset
Upload `students.csv` when running in Google Colab. If running locally, set `csv_path` to `data/students.csv`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
except ImportError:
    csv_path = "data/students.csv"

students = pd.read_csv(csv_path)
students.head()


## 2. Validate and inspect the data

In [ ]:
required = {"Student_ID", "Student_Name", *SUBJECTS, "Attendance"}
missing = required.difference(students.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if students["Student_ID"].duplicated().any():
    raise ValueError("Student_ID values must be unique.")

for col in SUBJECTS + ["Attendance"]:
    students[col] = pd.to_numeric(students[col], errors="coerce")
if students[SUBJECTS + ["Attendance"]].isna().any().any():
    raise ValueError("Marks and attendance must be numeric.")
if not students[SUBJECTS].apply(lambda s: s.between(0, 100)).all().all():
    raise ValueError("Marks must be between 0 and 100.")
if not students["Attendance"].between(0, 100).all():
    raise ValueError("Attendance must be between 0 and 100.")

print("Rows:", len(students))
display(students.describe(include="all"))


## 3. Calculate totals, averages, grades and results

In [ ]:
def assign_grade(p):
    if p >= 90: return "A+"
    if p >= 80: return "A"
    if p >= 70: return "B"
    if p >= 60: return "C"
    if p >= 50: return "D"
    if p >= 40: return "E"
    return "F"

analysis = students.copy()
analysis["Total_Marks"] = analysis[SUBJECTS].sum(axis=1)
analysis["Average_Marks"] = analysis[SUBJECTS].mean(axis=1).round(2)
analysis["Percentage"] = (analysis["Total_Marks"] / 400 * 100).round(2)
analysis["Grade"] = analysis["Percentage"].apply(assign_grade)
analysis["Result"] = np.where((analysis[SUBJECTS] >= PASS_MARK).all(axis=1), "Pass", "Fail")
analysis["Attendance_Status"] = np.where(
    analysis["Attendance"] >= ATTENDANCE_THRESHOLD, "Satisfactory", "Below threshold"
)
display(analysis.head(10))


## 4. Class summary and topper analysis

In [ ]:
print(f"Students analysed: {len(analysis)}")
print(f"Class average: {analysis['Percentage'].mean():.2f}%")
print("Pass:", (analysis["Result"] == "Pass").sum())
print("Fail:", (analysis["Result"] == "Fail").sum())
print(f"Average attendance: {analysis['Attendance'].mean():.2f}%")

top_five = analysis.nlargest(5, "Percentage")[
    ["Student_Name", "Percentage", "Grade", "Result"]
]
display(top_five)


## 5. Subject-wise performance

In [ ]:
subject_average = analysis[SUBJECTS].mean().round(2)
display(subject_average.to_frame("Average Marks"))

plt.figure(figsize=(8, 5))
subject_average.plot(kind="bar")
plt.title("Average Marks by Subject")
plt.ylabel("Average marks (out of 100)")
plt.xlabel("Subject")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


## 6. Grade distribution and attendance relationship

In [ ]:
grade_order = ["A+", "A", "B", "C", "D", "E", "F"]
grade_counts = analysis["Grade"].value_counts().reindex(grade_order, fill_value=0)

plt.figure(figsize=(7, 5))
grade_counts.plot(kind="bar")
plt.title("Grade Distribution")
plt.xlabel("Grade")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(analysis["Attendance"], analysis["Percentage"], alpha=0.8)
plt.title("Attendance vs Academic Percentage")
plt.xlabel("Attendance (%)")
plt.ylabel("Academic percentage (%)")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 7. Export results

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
analysis.to_csv(output_dir / "student_analysis_results.csv", index=False)
print("Saved:", output_dir / "student_analysis_results.csv")
